In [ ]:
!pip install antlr4-python3-runtime==4.9.3 --force-reinstall


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144554 sha256=6a8e2411126037f3676cb114b057be44a82e2614adbf23b450239a9312604a5f
  Stored in directory: /root/.cache/pip/wheels/1a/97/32/461f837398029ad76911109f07047fde1d7b661a147c7c56d1
Successfully built antlr4-python3-runtime
  Attempting uninstall: antlr4-python3-runtime
    Found existing installation: antlr4-python3-runtime 4.9.3
    Uninstalling antlr4-python3-runtime-4.9.3:
      Successfully uninstalled antlr4-python3-runtime-4.9.3


In [ ]:
grammar = """
lexer grammar CminusLexer;

KEYWORD
    : 'if'
    | 'else'
    | 'void'
    | 'int'
    | 'repeat'
    | 'break'
    | 'until'
    | 'return'
    ;

NUM : [0-9]+ ;

ID  : [a-zA-Z][a-zA-Z0-9]* ;

SYMBOL
    : '=='
    | ';' | ',' | '(' | ')' | '{' | '}' | '[' | ']'
    | '+' | '-' | '*' | '/' | '=' | '<'
    ;

WS              : [ \\t\\r\\n\\f]+ -> skip ;
LINE_COMMENT    : '//' ~[\\r\\n]* -> skip ;
BLOCK_COMMENT   : '/*' .*? '*/' -> skip ;
"""

with open("CminusLexer.g4", "w") as f:
    f.write(grammar)

print("✅ فایل CminusLexer.g4 بدون خطا و هماهنگ با خروجی tokens.txt ساخته شد.")

✅ فایل CminusLexer.g4 بدون خطا و هماهنگ با خروجی tokens.txt ساخته شد.


In [ ]:
# دانلود فایل antlr-4.13.1.jar
!wget -O antlr.jar https://www.antlr.org/download/antlr-4.9.3-complete.jar

# نصب antlr4-python3-runtime
!pip install antlr4-python3-runtime

# ساخت lexer از فایل CminusLexer.g4
!java -jar antlr.jar -Dlanguage=Python3 CminusLexer.g4


--2025-06-01 10:11:05--  https://www.antlr.org/download/antlr-4.9.3-complete.jar
Resolving www.antlr.org (www.antlr.org)... 185.199.111.153, 185.199.109.153, 185.199.110.153, ...
Connecting to www.antlr.org (www.antlr.org)|185.199.111.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3508089 (3.3M) [application/java-archive]
Saving to: ‘antlr.jar’

antlr.jar           100%[===================>]   3.34M  --.-KB/s    in 0.06s   

2025-06-01 10:11:06 (60.7 MB/s) - ‘antlr.jar’ saved [3508089/3508089]



In [ ]:
!java -jar antlr.jar -Dlanguage=Python3 CminusLexer.g4


In [ ]:
!ls -l CminusLexer*


-rw-r--r-- 1 root root  439 Jun  1 10:11 CminusLexer.g4
-rw-r--r-- 1 root root 3661 Jun  1 10:11 CminusLexer.interp
-rw-r--r-- 1 root root 3962 Jun  1 10:11 CminusLexer.py
-rw-r--r-- 1 root root   66 Jun  1 10:11 CminusLexer.tokens


In [ ]:
import re
from google.colab import files
import shutil

# 📥 مرحله 1: آپلود فایل
print("📥 لطفاً فایل input.txt را انتخاب و آپلود کنید:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
shutil.move(filename, "input.txt")
print("✅ فایل input.txt آماده شد.")

# 🔤 تعریف کلمات کلیدی، نمادها، برچسب‌ها
KEYWORDS = {'if', 'else', 'void', 'int', 'repeat', 'break', 'until', 'return'}
ANTLR_LABELS = {
    'if': 'KEYWORD', 'else': 'KEYWORD', 'void': 'KEYWORD', 'int': 'KEYWORD',
    'repeat': 'KEYWORD', 'break': 'KEYWORD', 'until': 'KEYWORD', 'return': 'KEYWORD',
    ';': 'SYMBOL', ',': 'SYMBOL', '(': 'SYMBOL', ')': 'SYMBOL',
    '{': 'SYMBOL', '}': 'SYMBOL', '[': 'SYMBOL', ']': 'SYMBOL',
    '+': 'SYMBOL', '-': 'SYMBOL', '*': 'SYMBOL', '/': 'SYMBOL',
    '=': 'SYMBOL', '<': 'SYMBOL', '==': 'SYMBOL'
}
SYMBOLS = sorted(['==', ';', ',', '(', ')', '{', '}', '[', ']', '+', '-', '*', '/', '=', '<'], key=lambda x: -len(x))

# 📄 مرحله 2: خواندن فایل و حذف کامنت‌ها
with open("input.txt", "r") as f:
    code = f.read()

code = re.sub(r'/\*.*?\*/', lambda m: ' ' * len(m.group()), code, flags=re.DOTALL)
code = re.sub(r'//.*', lambda m: ' ' * len(m.group()), code)

# بررسی unmatched comment
errors = []
for m in re.finditer(r'\*/', code):
    lineno = code[:m.start()].count('\n') + 1
    errors.append((lineno, '*/', 'Unmatched comment'))

unclosed = re.search(r'/\*.*$', code, flags=re.DOTALL)
if unclosed:
    lineno = code[:unclosed.start()].count('\n') + 1
    preview = unclosed.group(0)[:10].replace('\n', ' ')
    errors.append((lineno, preview + '...', 'Unclosed comment'))
    code = code.replace(unclosed.group(0), ' ' * len(unclosed.group(0)))

# 🧠 مرحله 3: تحلیل واژگانی
# 👇 پیش‌پردازش برای جداسازی ترکیب عدد و حروف مانند 23apple → 23 apple
code = re.sub(r'(\d+)([a-zA-Z_])', r'\1 \2', code)

tokens = []
symbol_table = {}
symbol_table = {}
keyword_set = set()  # ثبت keywordهایی که واقعاً در کد ظاهر شدن
id_order = {}        # ثبت شناسه‌ها با شماره خط اول وقوع

lines = code.splitlines()

for lineno, line in enumerate(lines, start=1):
    i = 0
    while i < len(line):
        chunk = line[i:]

        if re.match(r'^[ \t\r\f\v]', chunk):
            i += 1
            continue

        # عدد صحیح
        # عدد صحیح با بررسی نامعتبر بودن مثل 3d یا 45abc
        num = re.match(r'^\d+', chunk)
        if num:
            value = num.group()
            rest = chunk[len(value):]
            if rest and re.match(r'^[a-zA-Z_]', rest[0]):
                # مانند: 3d → عدد نامعتبر
                error_token = value + rest[0]
                errors.append((lineno, error_token, 'Invalid number'))
                i += len(error_token)
                continue
            tokens.append((lineno, 'NUM', value))
            i += len(value)
            continue



        # شناسه یا کلیدواژه
        ident = re.match(r'^[A-Za-z_][A-Za-z0-9_]*', chunk)
        if ident:
            value = ident.group()
            if value in KEYWORDS:
                tokens.append((lineno, 'KEYWORD', value))
                if value not in keyword_set:
                    keyword_set.add(value)
            else:
                tokens.append((lineno, 'ID', value))
                if value not in id_order:
                    id_order[value] = lineno
            i += len(value)
            continue




        # نمادها مثل == یا + و ...
        matched = False
        for sym in SYMBOLS:
            if chunk.startswith(sym):
                tokens.append((lineno, 'SYMBOL', sym))
                i += len(sym)
                matched = True
                break
        if matched:
            continue

        # کاراکتر نامعتبر
        errors.append((lineno, chunk[0], 'Invalid input'))
        i += 1

# 📄 مرحله 4: خروجی به فایل‌ها

# توکن‌ها
with open("tokens.txt", "w") as f:
    current_line = -1
    for lineno, typ, val in tokens:
        if lineno != current_line:
            if current_line != -1:
                f.write("\n")
            f.write(f"{lineno}.\t")
            current_line = lineno
        f.write(f"({typ}, {val}) ")

errors.sort(key=lambda x: x[0])
# خطاهای واژگانی
with open("lexical_errors.txt", "w") as f:
    if not errors:
        f.write("There is no lexical error.")
    else:
        current_line = -1
        for lineno, val, msg in errors:
            if lineno != current_line:
                if current_line != -1:
                    f.write("\n")
                f.write(f"{lineno}.\t")
                current_line = lineno
            f.write(f"({val}, {msg}) ")


with open("symbol_table.txt", "w") as f:
    all_keywords = ['break', 'else', 'if', 'int', 'repeat', 'return', 'until', 'void']
    output = []

    for kw in all_keywords:
        output.append(kw)

    # بعد: شناسه‌ها به ترتیب اولین وقوع‌شان در کد
    sorted_ids = sorted(id_order.items(), key=lambda x: x[1])
    for sym, _ in sorted_ids:
        output.append(sym)

    # چاپ
    for i, sym in enumerate(output, start=1):
        f.write(f"{i}.\t{sym}\n")


print("✅ تحلیل واژگانی کامل شد و فایل‌های خروجی ساخته شدند.")

📥 لطفاً فایل input.txt را انتخاب و آپلود کنید:


Saving input.txt to input.txt
✅ فایل input.txt آماده شد.
✅ تحلیل واژگانی کامل شد و فایل‌های خروجی ساخته شدند.


In [ ]:
import zipfile

with zipfile.ZipFile("compiler_outputs.zip", "w") as zipf:
    zipf.write("tokens.txt")
    zipf.write("lexical_errors.txt")
    zipf.write("symbol_table.txt")

files.download("compiler_outputs.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# ✅ حذف ماژول قبلی از حافظه در صورت وجود
import sys
if "CminusLexer" in sys.modules:
    del sys.modules["CminusLexer"]

# ✅ افزودن مسیر ماژول ساخته‌شده توسط ANTLR
sys.path.insert(0, "/content/antlr_gen")

# ✅ ایمپورت lexer از مسیر درست
from antlr4 import *
from CminusLexer import CminusLexer

# ✅ اجرای lexer و تولید خروجی توکن‌ها در فایل 1p_ANTLR.txt
def run_antlr_tokenizer():
    with open("input.txt") as f:
        input_data = f.read()

    input_stream = InputStream(input_data)
    lexer = CminusLexer(input_stream)
    tokens = CommonTokenStream(lexer)
    tokens.fill()

    with open("1p_ANTLR.txt", "w") as out:
        current_line = -1
        for token in tokens.tokens:
            if token.type == Token.EOF:
                break
            line = token.line
            text = token.text
            typ = lexer.symbolicNames[token.type]
            if line != current_line:
                if current_line != -1:
                    out.write("\n")
                out.write(f"{line}.\t")
                current_line = line
            out.write(f"({typ}, {text}) ")

run_antlr_tokenizer()
print("✅ فایل 1p_ANTLR.txt با موفقیت ساخته شد.")


✅ فایل 1p_ANTLR.txt با موفقیت ساخته شد.


line 9:10 token recognition error at: '!'
line 14:15 token recognition error at: '@'


In [ ]:
import re

def Check(verbose=True):
    def extract_tokens(path):
        tokens = []
        with open(path) as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) != 2:
                    continue
                token_pairs = re.findall(r'\([A-Z]+, [^)]+\)', parts[1])
                tokens.extend(token_pairs)
        return tokens

    my_tokens = extract_tokens("tokens.txt")
    antlr_tokens = extract_tokens("1p_ANTLR.txt")

    min_len = min(len(my_tokens), len(antlr_tokens))
    match_count = 0
    mismatches = []

    for i in range(min_len):
        if my_tokens[i] == antlr_tokens[i]:
            match_count += 1
        else:
            mismatches.append((i, my_tokens[i], antlr_tokens[i]))

    percent = (match_count / min_len) * 100 if min_len > 0 else 0
    print(f"✅ درصد تطابق بین توکن‌های اسکنر شما و ANTLR: {percent:.2f}%")

    if verbose:
        print(f"\n🔍 تعداد توکن‌های برابر: {match_count} از {min_len}")
        print(f"❌ تعداد تفاوت‌ها: {len(mismatches)}\n")

        if mismatches:
            print("📌 لیست کامل تفاوت‌ها:")
            for idx, mine, antlr in mismatches:
                print(f"  [{idx}] mine: {mine} | antlr: {antlr}")
Check()

✅ درصد تطابق بین توکن‌های اسکنر شما و ANTLR: 100.00%

🔍 تعداد توکن‌های برابر: 62 از 62
❌ تعداد تفاوت‌ها: 0



In [ ]:
!apt-get update
!apt-get install -y openjdk-11-jdk
!pip install antlr4-tools


Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,721 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,953 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [8,994 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://security.ubuntu.com/ubuntu

In [ ]:
parser_grammar = """
parser grammar CminusParser;

options { tokenVocab = CminusLexer; }

program : declaration_list ;

declaration_list
    : declaration_list declaration
    | /* epsilon */ ;

declaration
    : declaration_initial declaration_prime ;

declaration_initial
    : type_specifier ID ;

declaration_prime
    : var_declaration_prime
    | fun_declaration_prime ;

var_declaration_prime
    : SYMBOL NUM SYMBOL SYMBOL     // [ NUM ] ;
    | SYMBOL                       // ;
    ;

fun_declaration_prime
    : SYMBOL params SYMBOL compound_stmt // ( params ) { ... }
    ;

type_specifier
    : KEYWORD                      // int | void
    ;

params
    : param (SYMBOL param)*        // param , param , ...
    | KEYWORD                      // void
    ;

param
    : type_specifier ID param_prime ;

param_prime
    : SYMBOL SYMBOL                // [ ]
    | /* epsilon */ ;

compound_stmt
    : SYMBOL declaration_list statement_list SYMBOL // { decls stmts }
    ;

statement_list
    : statement_list statement
    | /* epsilon */ ;

statement
    : expression_stmt
    | compound_stmt
    | selection_stmt
    | iteration_stmt
    | return_stmt ;

expression_stmt
    : expression SYMBOL
    | KEYWORD SYMBOL               // break ;
    | SYMBOL                       // ;
    ;

selection_stmt
    : KEYWORD SYMBOL expression SYMBOL statement KEYWORD statement
    // if (expr) stmt else stmt
    ;

iteration_stmt
    : KEYWORD statement KEYWORD SYMBOL expression SYMBOL
    // repeat stmt until (expr)
    ;

return_stmt
    : KEYWORD return_stmt_prime ;

return_stmt_prime
    : SYMBOL                       // ;
    | expression SYMBOL
    ;

expression
    : simple_expression_zegond
    | ID b ;

b
    : SYMBOL expression                // =
    | SYMBOL expression SYMBOL h       // [ expr ] =
    | simple_expression_prime ;

h
    : SYMBOL expression                // =
    | g d c ;

simple_expression_zegond
    : additive_expression_zegond c ;

simple_expression_prime
    : additive_expression_prime c ;

c
    : relop additive_expression
    | /* epsilon */ ;

relop
    : SYMBOL                          // < or ==
    ;

additive_expression
    : term d ;

additive_expression_prime
    : term_prime d ;

additive_expression_zegond
    : term_zegond d ;

d
    : addop term d
    | /* epsilon */ ;

addop
    : SYMBOL                          // + or -
    ;

term
    : factor g ;

term_prime
    : factor_prime g ;

term_zegond
    : factor_zegond g ;

g
    : SYMBOL factor g                 // * factor g
    | /* epsilon */ ;

factor
    : SYMBOL expression SYMBOL        // ( expr )
    | ID var_call_prime
    | NUM ;

var_call_prime
    : SYMBOL args SYMBOL              // ( args )
    | var_prime ;

var_prime
    : SYMBOL expression SYMBOL        // [ expr ]
    | /* epsilon */ ;

factor_prime
    : SYMBOL args SYMBOL              // ( args )
    | /* epsilon */ ;

factor_zegond
    : SYMBOL expression SYMBOL        // ( expr )
    | NUM ;

args
    : arg_list
    | /* epsilon */ ;

arg_list
    : expression arg_list_prime ;

arg_list_prime
    : SYMBOL expression arg_list_prime  // , expr ...
    | /* epsilon */ ;


"""

with open("CminusParser.g4", "w") as f:
    f.write(parser_grammar)

print("✅ فایل CminusParser.g4 نوشته شد.")


✅ فایل CminusParser.g4 نوشته شد.


In [ ]:
!pip install antlr4-python3-runtime==4.13.1 --force-reinstall


--2025-06-01 10:12:02--  https://www.antlr.org/download/antlr-4.9.3-complete.jar
Resolving www.antlr.org (www.antlr.org)... 185.199.110.153, 185.199.108.153, 185.199.109.153, ...
Connecting to www.antlr.org (www.antlr.org)|185.199.110.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3508089 (3.3M) [application/java-archive]
Saving to: ‘antlr-4.9.3-complete.jar’

antlr-4.9.3-complet 100%[===================>]   3.34M  --.-KB/s    in 0.06s   

2025-06-01 10:12:02 (58.1 MB/s) - ‘antlr-4.9.3-complete.jar’ saved [3508089/3508089]



In [ ]:
!rm CminusLexer.py CminusParser.py CminusParserListener.py CminusParser.tokens CminusLexer.tokens


In [ ]:
# دانلود jar
!wget https://www.antlr.org/download/antlr-4.13.1-complete.jar -O antlr-4.13.1-complete.jar

# نصب پکیج پایتون


# تولید lexer و parser
!java -Xmx500M -cp antlr-4.13.1-complete.jar org.antlr.v4.Tool -Dlanguage=Python3 CminusLexer.g4
!java -Xmx500M -cp antlr-4.13.1-complete.jar org.antlr.v4.Tool -Dlanguage=Python3 CminusParser.g4


--2025-06-01 10:19:57--  https://www.antlr.org/download/antlr-4.13.1-complete.jar
Resolving www.antlr.org (www.antlr.org)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to www.antlr.org (www.antlr.org)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2139203 (2.0M) [application/java-archive]
Saving to: ‘antlr-4.13.1-complete.jar’

antlr-4.13.1-comple 100%[===================>]   2.04M  --.-KB/s    in 0.07s   

2025-06-01 10:19:57 (27.7 MB/s) - ‘antlr-4.13.1-complete.jar’ saved [2139203/2139203]



In [ ]:
from antlr4 import *
from CminusLexer import CminusLexer
from CminusParser import CminusParser
from CminusParserListener import CminusParserListener


In [ ]:
from google.colab import files
uploaded = files.upload()

for name in uploaded:
    input_file_path = name
    print(f"📥 فایل ورودی {input_file_path} بارگذاری شد.")


Saving input.txt to input (1).txt
📥 فایل ورودی input (1).txt بارگذاری شد.


In [ ]:
from antlr4 import *
from antlr4.tree.Tree import TerminalNodeImpl
from CminusLexer import CminusLexer
from CminusParser import CminusParser

recursive_rules = [
    "declaration_list",
    "statement_list",
    "g"
    # D رو جدا و خاص هندل می‌کنیم پایین
]

class FullParseTreePrinter:
    def __init__(self, parser):
        self.parser = parser
        self.lines = []

    def print_node(self, ctx, ancestors=[], is_last=True, is_root=False):
        indent = ""
        for ancestor_last in ancestors:
            indent += "    " if ancestor_last else "│   "
        connector = "└── " if is_last else "├── "
        prefix = indent + connector

        # ریشه
        if is_root:
            if hasattr(ctx, "getRuleIndex"):
                rule_index = ctx.getRuleIndex()
                rule_name = self.parser.ruleNames[rule_index].capitalize().replace("_", "-")
                self.lines.append(rule_name)
            children = [ctx.getChild(i) for i in range(ctx.getChildCount())]
            for i, child in enumerate(children):
                is_last_child = (i == len(children) - 1)
                self.print_node(child, ancestors=[], is_last=is_last_child)
            # محاسبه عمق خطوط عمودی برای $
            last_indent = 0
            for c in reversed(self.lines):
                if c.strip():
                    last_indent = c.count("│")
                    break
            for _ in range(last_indent):
                self.lines.append("│")
            self.lines.append("└── $")
            return

        # ترمینال
        if isinstance(ctx, TerminalNodeImpl):
            symbol = ctx.getSymbol()
            token_type = symbol.type
            token_name = self.parser.symbolicNames[token_type]
            token_text = symbol.text
            if token_name is None or token_text.strip() == "":
                return
            self.lines.append(prefix + f"({token_name}, {token_text})")
            return

        if not hasattr(ctx, "getRuleIndex"):
            return

        rule_index = ctx.getRuleIndex()
        rule_name = self.parser.ruleNames[rule_index]
        display_name = rule_name.capitalize().replace("_", "-")
        self.lines.append(prefix + display_name)

        children = [ctx.getChild(i) for i in range(ctx.getChildCount())]

        # مخصوص D: ساختار بازگشتی استاد
        if rule_name == "d":
            if len(children) == 0:
                epsilon_indent = indent + ("    " if is_last else "│   ")
                self.lines.append(epsilon_indent + "└── epsilon")
                return
            elif len(children) == 3:
                self.print_node(children[0], ancestors + [False], False)
                self.print_node(children[1], ancestors + [False, False], False)
                self.print_node(children[2], ancestors + [False, False, True], True)
                return
            elif len(children) == 2:
                self.print_node(children[0], ancestors + [False], False)
                self.print_node(children[1], ancestors + [False, False], False)
                # اضافه کردن نود D و زیرش epsilon مثل استاد
                extra_indent = indent + "│   "
                self.lines.append(extra_indent + "└── D")
                self.lines.append(extra_indent + "    └── epsilon")
                return

        # بازسازی بازگشتی برای سایر لیست‌های بازگشتی چپ
        if rule_name in recursive_rules:
            if len(children) == 1:
                self.print_node(children[0], ancestors + [False], False)
                epsilon_indent = indent + "│   "
                self.lines.append(epsilon_indent + "└── " + display_name)
                self.lines.append(epsilon_indent + "    └── epsilon")
                return
            # برای G اگر 2 بچه داره (عملگر و عامل)، نود بازگشتی و epsilon اضافه کن
            if rule_name in ["g"] and len(children) == 2:
                self.print_node(children[0], ancestors + [False], False)
                self.print_node(children[1], ancestors + [False, False], False)
                epsilon_indent = indent + "│   "
                self.lines.append(epsilon_indent + f"└── {display_name}")
                self.lines.append(epsilon_indent + "    └── epsilon")
                return

        if len(children) == 0:
            epsilon_indent = indent + ("    " if is_last else "│   ")
            self.lines.append(epsilon_indent + "└── epsilon")
            return

        for i, child in enumerate(children):
            is_last_child = (i == len(children) - 1)
            self.print_node(child, ancestors + [is_last], is_last_child)

def generate_parse_tree(input_code_path):
    input_stream = FileStream(input_code_path)
    lexer = CminusLexer(input_stream)
    tokens = CommonTokenStream(lexer)
    parser = CminusParser(tokens)

    tree = parser.program()

    printer = FullParseTreePrinter(parser)
    printer.print_node(tree, is_root=True)

    # حذف خطوط عمودی اضافه قبل از $
    while len(printer.lines) >= 2 and printer.lines[-1] == "└── $" and printer.lines[-2] == "└── $":
        printer.lines.pop(-2)

    with open("parse_tree.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(printer.lines))

    print("✅ خروجی parse_tree.txt دقیقاً مثل استاد ساخته شد.")


In [ ]:
import sys
from antlr4.error.ErrorListener import ErrorListener

class SyntaxErrorListener(ErrorListener):
    def __init__(self):
        self.errors = []

    def syntaxError(self, recognizer, offendingSymbol, line, column, msg, e):
        self.errors.append(f"Syntax Error at line {line}:{column} - {msg}")

def generate_errors(input_code):
    input_stream = FileStream(input_code)
    lexer = CminusLexer(input_stream)
    stream = CommonTokenStream(lexer)
    parser = CminusParser(stream)

    error_listener = SyntaxErrorListener()
    parser.removeErrorListeners()
    parser.addErrorListener(error_listener)

    parser.program()  # اجرای parser

    with open("syntax_errors.txt", "w") as f:
        if error_listener.errors:
            f.write("\n".join(error_listener.errors))
        else:
            f.write("There is no syntax error.")

    print("❗ فایل syntax_errors.txt ساخته شد.")

generate_errors(input_file_path)


❗ فایل syntax_errors.txt ساخته شد.


In [ ]:
# تولید parse tree
generate_parse_tree(input_file_path)



# دانلود فایل‌ها
from google.colab import files
files.download("parse_tree.txt")



✅ خروجی parse_tree.txt دقیقاً مثل استاد ساخته شد.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>